# Utiliser des modèles via leur API 🔌

Dans cette leçon, nous allons nous connecter depuis Python à des LLM hébergés.

Nous prendrons Google AI Studio comme exemple, principalement parce que son offre gratuite permet de tester facilement, mais les mêmes principes s'appliquent aux autres fournisseurs de LLM.

Le modèle s'exécute sur les serveurs de Google ; Python lui envoie une requête par Internet et reçoit une réponse.


## 1. Que se passe-t-il lorsque nous appelons une API de LLM ?

La requête contient une clé API pour l'authentification, l'identifiant d'un modèle et une liste de messages. Google AI Studio renvoie une réponse contenant le texte produit par le modèle et des métadonnées, telles que le nombre de jetons utilisés.

Le paquet Python officiel `google-generative-ai` est un SDK : il fournit des objets et des méthodes Python qui construisent ces requêtes API pour nous. Plus tard, nous enverrons directement une requête au point de terminaison HTTP afin de voir ce que fait le SDK.


## 2. Créer un compte et une clé API 🔐

1. Rendez-vous sur [Google AI Studio](https://aistudio.google.com/apikey). Connectez-vous ou créez un compte.

2. Choisissez **New key** et copiez la clé générée. Attention : une fois la fenêtre fermée, vous ne pourrez plus la consulter. Si vous la perdez, vous devrez la révoquer et en créer une nouvelle.

3. Collez la clé dans un nouveau fichier nommé `.env` dans le dossier courant (le point initial rend le fichier caché sous Linux et macOS). Le fichier doit contenir une seule ligne, comme ceci :

```
GEMINI_API_KEY="YOUR_KEY_HERE"
```

Si vous utilisez Git, n'oubliez pas d'ajouter `.env` à votre fichier `.gitignore` afin d'éviter de le publier par erreur.

> **AVERTISSEMENT** : une clé est un **identifiant secret**. Ne la collez jamais dans une cellule du notebook, ne la commitez pas dans Git, ne l'incluez pas dans une capture d'écran et ne la partagez pas dans un message. Si elle est divulguée, révoquez-la dans Studio et créez-en une nouvelle.


## 3. Installer le SDK Python et dotenv 📦

Nous aurons besoin du paquet `google-genai` pour appeler l'API et de `python-dotenv` pour lire la clé API depuis le fichier `.env`.

Vous pouvez installer ces paquets avec pip ou uv.


#### Avec pip 📦


#### Ou avec uv 📦

Ouvrez un terminal, placez-vous dans le dossier du projet et exécutez :

```bash
uv add google-genai python-dotenv
```


### Rendre la clé accessible à Python 🔐

Utilisons le paquet `python-dotenv` pour lire le fichier `.env` et définir la clé comme variable d'environnement. Nous pourrons ainsi l'utiliser en Python sans l'inscrire directement dans le code.


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
api_key = os.environ.get("GEMINI_API_KEY")

## Choisir votre modèle 🤖

La liste des modèles disponibles se trouve sur https://aistudio.google.com/docs/models. Pour cette leçon, nous utiliserons `gemini-3.5-flash-lite`.


In [ ]:
# Let's create a client, and a string that contains the model's name.
from google import genai
client = genai.Client(api_key=api_key)
MODEL = "gemini-3.5-flash-lite"

In [ ]:
# First API call
response = client.models.generate_content(
    model=MODEL, contents='Why is the sky blue?'
)
print(response.text[:300])

In [ ]:
response

L'objet `response` contient davantage que la réponse du modèle. Examinez les métadonnées du modèle et de l'utilisation. Le nombre de jetons permet de comprendre la taille de la requête ; il ne donne pas nécessairement le quota gratuit exact qu'il vous reste.


### Combien coûte l'utilisation d'un LLM ? 💰

Google AI Studio propose une offre gratuite qui limite le nombre de requêtes par minute, par jour et par mois. Si vous dépassez ces limites, votre requête ne sera pas traitée et vous rencontrerez une erreur 429 : « Resource Exhausted ».

[Consulter les limites de requêtes](https://aistudio.google.com/rate-limit)


### Paramètres ⚙️

#### Rôle, échantillonnage et reproductibilité 🎛️

* **`system_instruction`** : définit une persona, des règles, un ton ou des contraintes de sortie persistantes pour la requête.
* **`temperature`** : contrôle le caractère aléatoire/créatif du choix des jetons (une valeur faible donne un résultat plus déterministe et ciblé ; une valeur élevée, des résultats plus variés). La température se situe le plus souvent entre 0 et 2. Elle ne garantit pas un [résultat déterministe](https://medium.com/google-cloud/is-a-zero-temperature-deterministic-c4a7) : même avec `temperature=0`, le modèle peut produire des résultats différents pour la même consigne.
* **`top_p`** *(échantillonnage par noyau)* : limite la sélection au plus petit ensemble de jetons dont la probabilité cumulée atteint `top_p` (par exemple `0.9`). Ce paramètre est souvent réglé à la place de `temperature`, ou en complément. Exemple : `top_p=0.9` signifie « choisir parmi le plus petit ensemble de jetons dont la probabilité cumulée atteint 90 % ».
* **`top_k`** : limite la sélection aux `top_k` jetons les plus probables (par exemple `40`). Une valeur faible rend le résultat plus ciblé ; une valeur élevée le rend plus varié. Exemple : `top_k=40` signifie « choisir parmi les 40 jetons les plus probables ».
* **`seed`** : fixe la graine aléatoire pour rendre les réponses plus reproductibles dans des conditions identiques (de façon indicative). Exemple : `seed=42` signifie « utiliser 42 comme graine aléatoire ». Cela ne garantit toutefois pas des réponses identiques avec des versions de modèles ou du matériel différents.

#### Longueur et arrêt ⏱️

* **`max_output_tokens`** : limite le nombre maximal de jetons générés (ce qui influe sur la latence, la longueur et le coût). Lorsque cette limite est atteinte, la génération s'arrête ; la phrase peut donc être tronquée.
* **`stop_sequences`** : chaînes de caractères (par exemple `['---', 'END']`) qui interrompent immédiatement la génération lorsqu'elles sont produites.

### 3. Sortie structurée, raisonnement, outils et sécurité 🧩

* **`response_mime_type` et `response_schema`** : imposent une sortie structurée (par exemple `application/json` avec un modèle Pydantic ou un schéma JSON).
* **`thinking_config`** : ajuste l'effort ou le budget de raisonnement des modèles capables de raisonner (compromis entre profondeur et latence).
* **`tools` et `tool_config`** : enregistrent des fonctions, des sources de données ou l'exécution de code, et définissent le mode d'invocation (`AUTO`, `ANY`, `NONE`).
* **`safety_settings`** : définit les seuils des filtres de contenu selon les catégories de risques.


### Exemple rapide 💬


In [ ]:
from google import genai
from google.genai import types
from pydantic import BaseModel


class Summary(BaseModel):
    overview: str
    key_points: list[str]

response = client.models.generate_content(
    model=MODEL,
    contents="Explain why Marseille is worth visiting",
    config=types.GenerateContentConfig(
        system_instruction="You are a travel expert.",
        temperature=0.3,
        top_p=0.9,
        max_output_tokens=300,
        stop_sequences=["\n\n\n"],
        response_mime_type="application/json",
        response_schema=Summary,
    ),
)

print(response.text)


### Aide-mémoire 📋

| Objectif | Paramètre(s) clé(s) |
| --- | --- |
| Définir le rôle et les contraintes globales | `system_instruction` |
| Imposer une sortie JSON valide | `response_mime_type` + `response_schema` |
| Limiter la longueur et le coût | `max_output_tokens`, `stop_sequences` |
| Améliorer la cohérence | `temperature` faible + `top_p` modéré + `seed` |
| Connecter des fonctions ou données externes | `tools`, `tool_config` |


## Température, top-p, top-k et seed 🌡️

Testons maintenant différentes valeurs de température, top-p, top-k et seed pour observer leur effet sur la réponse du modèle.


In [ ]:
from time import sleep

def chat(messages="""Give me 3 superpowers that would be interesting to have but are completely useless in everyday life.
         Each answer must be a short sentence (maximum 10 words). Avoid repeating ideas.""",
         temperature=1.0):
    response = client.models.generate_content(
        model=MODEL,
        contents=messages,
        config=types.GenerateContentConfig(
            system_instruction="You are a helpful assistant.",
            temperature=temperature,
            max_output_tokens=60,
            stop_sequences=["\n\n\n"],
            seed=42,
            top_p=0.95
        ),
    )
    return response.text

count = 0
for temperature in (0, 0, 1, 1, 2, 2):
    count += 1
    print(f"temperature={temperature}: \n{chat(temperature=temperature)}")
    if count % 2 == 0: print("\n --- \n")  # Print a separator after every 2 requests
    sleep(1)  # Sleep for 1 second to avoid rate limiting